# End-sem 06. Explanations, case studies and the dashboard screenshots
SHAP on the ranker, the translated radar, three hits and one bad miss. The same figures are drawn live by `app/shadow_squad.py`.

In [1]:
import sys, os, subprocess, time; sys.path.insert(0, 'src')
import pandas as pd, numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt, shap
from common import *
import engine as E, viz
lake = E.Lake(); models = E.load_models(); PF = pd.read_parquet('lake/pairs.parquet'); LG = pd.read_parquet('lake/scores_lgbm.parquet'); EV = pd.read_parquet('lake/replacement_events')
EV['qid'] = EV['C'].astype(str) + '_' + EV['t'].astype(str) + '_' + EV['S'].astype(str); M = {}

/home/srioo/Desktop/Projects/root/BigP/bigdata/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. What drives the ranking overall (SHAP, out-of-time rows only)

In [2]:
parts = []
for Y in (2020, 2021, 2022, 2023):
    X = PF[(PF['t'] == Y) & PF['q_in_pool'] & PF['candidate']]; parts.append((shap.TreeExplainer(models[Y])(X[E.FEATS]).values, X))
SV = np.vstack([p[0] for p in parts]); XA = pd.concat([p[1] for p in parts]); imp = pd.Series(np.abs(SV).mean(0), index=E.FEATS).sort_values()
dirn = {f: float(np.corrcoef(XA[f].fillna(XA[f].median()), SV[:, i])[0, 1]) for i, f in enumerate(E.FEATS)}
fig, ax = plt.subplots(figsize=(6, 3.2)); ax.barh([viz.NICE[f] for f in imp.index], imp.values, color=['#d95f02' if dirn[f] > 0 else '#1f77b4' for f in imp.index]); ax.set_xlabel('mean |SHAP| on unseen events (orange: higher value raises the rank, blue: lowers it)'); ax.tick_params(labelsize=7)
plt.tight_layout(); plt.savefig('outputs/figs/p6_shap_global.png', dpi=160); plt.close()
M['shap_global'] = {'mean_abs_shap': imp.round(4).to_dict(), 'direction_corr': {k: round(v, 3) for k, v in dirn.items()}, 'rows': len(XA)}; print(M['shap_global']['mean_abs_shap'])

{'traj_sim': 0.1555, 'sim_translated': 0.248, 'pipeline_rank': 0.2879, 'composite_perf_index': 0.2991, 'value_gap': 0.3149, 'club_taste': 0.323, 'log_value': 0.4114, 'age_num': 0.4169, 'hazard': 0.421}


## 2. Case studies: three hits and one miss
Chosen by rule, not by hand: the three unseen events with the best rank for the real signing (ties broken by the larger fee), and the unseen event where a real signing inside the LSH candidate set was ranked worst.

In [3]:
R = LG.merge(PF[['qid', 'player_id', 'candidate']], on=['qid', 'player_id']); R['rank'] = R.groupby('qid')['score'].rank(ascending=False, method='first'); R['pool'] = R.groupby('qid')['score'].transform('size')
tr = R[R['y'] == 1].merge(EV[['qid', 'N', 'club', 't', 'S_name', 'P', 'N_name', 'N_from', 'fee', 'C', 'S', 'league']].rename(columns={'N': 'player_id', 't': 't_'}), on=['qid', 'player_id'])
hits = tr[tr['rank'] <= 10].sort_values(['rank', 'fee'], ascending=[True, False]).drop_duplicates('qid').head(3); miss = tr[tr['candidate']].sort_values('rank', ascending=False).head(1); cases = []
for kind, row in [('hit', r) for r in hits.itertuples()] + [('miss', r) for r in miss.itertuples()]:
    X = E.rank(lake, models, row.C, row.t, row.S, row.P, top=10 ** 6); X['rank'] = range(1, len(X) + 1); me = X[X['player_id'] == row.player_id].iloc[0]; tag = f"{kind}_{row.club.replace(' ', '')}_{row.t}_{row.N_name.split()[-1]}"
    pct = {f: float((X[f] < me[f]).mean()) for f in E.FEATS}
    for who, lab_ in ((me, 'signed'), (X.iloc[0], 'top1')):
        viz.radar(lake, who, row.S, row.t).savefig(f'outputs/figs/case_{tag}_{lab_}_radar.png', dpi=150); plt.close('all'); viz.waterfall(models[row.t], X, int(who.name)).savefig(f'outputs/figs/case_{tag}_{lab_}_shap.png', dpi=150); plt.close('all')
    cases.append({'kind': kind, 'tag': tag, 'club': row.club, 'league': row.league, 'season': int(row.t), 'starter': row.S_name, 'position': row.P, 'signed': row.N_name, 'from': row.N_from, 'fee': None if pd.isna(row.fee) else float(row.fee), 'rank': int(me['rank']), 'pool': len(X),
                  'signed_feature_percentile_in_pool': {k: round(v, 2) for k, v in pct.items()}, 'signed_values': {f: (None if pd.isna(me[f]) else round(float(me[f]), 3)) for f in E.FEATS}, 'starter_value': None if pd.isna(me['S_value']) else float(me['S_value']), 'signed_value': float(me['value']),
                  'top5': X.head(5)[['player', 'squad', 'age_num', 'value', 'score']].round(3).to_dict('records'), 'url': f"?league={row.league}&club={row.club}&t={row.t}&S={row.S}&X={row.player_id}".replace(' ', '%20')})
    print(f"{kind.upper()}: {row.club} {row.t}-{str(row.t + 1)[2:]}, {row.S_name} ({row.P}) -> {row.N_name} from {row.N_from}; ranked {int(me['rank'])} of {len(X)}; percentiles {cases[-1]['signed_feature_percentile_in_pool']}")
M['cases'] = cases

HIT: Dortmund 2023-24, Niclas Füllkrug (ST) -> Maximilian Beier from Hoffenheim; ranked 2 of 76; percentiles {'sim_translated': 0.09, 'traj_sim': 0.04, 'pipeline_rank': 0.86, 'club_taste': 0.33, 'hazard': 0.91, 'log_value': 0.88, 'value_gap': 0.46, 'age_num': 0.05, 'composite_perf_index': 0.64}


HIT: Paris S-G 2022-23, Neymar (AM) -> Kang-in Lee from RCD Mallorca; ranked 2 of 42; percentiles {'sim_translated': 0.93, 'traj_sim': 0.81, 'pipeline_rank': 0.24, 'club_taste': 0.76, 'hazard': 0.95, 'log_value': 0.74, 'value_gap': 0.24, 'age_num': 0.17, 'composite_perf_index': 0.45}


HIT: Paris S-G 2021-22, Georginio Wijnaldum (CM) -> Carlos Soler from Valencia; ranked 2 of 104; percentiles {'sim_translated': 0.6, 'traj_sim': 0.63, 'pipeline_rank': 0.66, 'club_taste': 0.51, 'hazard': 0.98, 'log_value': 0.97, 'value_gap': 0.54, 'age_num': 0.53, 'composite_perf_index': 0.75}


MISS: Marseille 2022-23, Sead Kolašinac (CB) -> Bamo Meïté from FC Lorient; ranked 131 of 133; percentiles {'sim_translated': 0.1, 'traj_sim': 0.14, 'pipeline_rank': 0.73, 'club_taste': 0.17, 'hazard': 0.56, 'log_value': 0.02, 'value_gap': 0.77, 'age_num': 0.02, 'composite_perf_index': 0.08}


## 3. Dashboard screenshots (taken from the running Streamlit app)

In [4]:
import json
json.dump([{'tag': c['tag'], 'url': c['url']} for c in cases], open('outputs/screens/cases.json', 'w'))
out = subprocess.run([sys.executable, 'src/screens.py', 'outputs/screens/cases.json'], capture_output=True, text=True); print(out.stderr[-800:]); shots = out.stdout.split()
assert len(shots) == len(cases), 'screenshots missing'
M['screens'] = shots; print(shots); _ = put_metrics('phase6', M)


['outputs/screens/hit_Dortmund_2023_Beier.png', 'outputs/screens/hit_ParisS-G_2022_Lee.png', 'outputs/screens/hit_ParisS-G_2021_Soler.png', 'outputs/screens/miss_Marseille_2022_Meïté.png']
